In [2]:
!pip install torch torchvision datasets pillow numpy tqdm diffusers timm accelerate


# ELT Training Notebook
This notebook runs the ELT training loop on a single GPU without the DDP (DistributedDataParallel) environment overhead. It's designed to be simple and easy to run.

In [3]:
import os
import sys
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
from torchvision import transforms
from PIL import Image
import numpy as np
from copy import deepcopy
from collections import OrderedDict
from tqdm.auto import tqdm
from diffusers.models import AutoencoderKL

# Add ELT and DiT paths to sys.path
sys.path.append(os.path.abspath('..'))
sys.path.append(os.path.abspath('../dit-model-implementation/dit'))

from ELT.configs.config import get_config
from ELT.models.elt import ELT
from ELT.diffusion import create_diffusion
from ELT.train import center_crop_arr, CIFAR10Dataset, update_ema, requires_grad


In [4]:
# 1. Setup Configuration & Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
config = get_config()
print(f'Using device: {device}')

# 2. Initialize Models
model = ELT(config).to(device)
ema = deepcopy(model).to(device)
requires_grad(ema, False)
update_ema(ema, model, decay=0)

diffusion = create_diffusion(timestep_respacing='')
vae = AutoencoderKL.from_pretrained('stabilityai/sd-vae-ft-ema').to(device)

print(f'ELT Parameters: {sum(p.numel() for p in model.parameters()):,}')


Using device: cuda


/opt/watchdog/users/parthbrijpuria/elt-project/ELT/.venv/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


ELT Parameters: 7,380,512


In [4]:
# 3. Setup Dataset and DataLoader
transform = transforms.Compose([
    transforms.Lambda(lambda pil_image: center_crop_arr(pil_image, config.model.image_size)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5], inplace=True)
])
hf_dataset = load_dataset('uoft-cs/cifar10', split='train')
dataset = CIFAR10Dataset(hf_dataset, transform=transform)
loader = DataLoader(
    dataset,
    batch_size=128,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    drop_last=True
)
print(f'Dataset contains {len(dataset):,} images')


Dataset contains 50,000 images


In [5]:
# 4. Training Loop
opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0)
epochs = 600
results_dir = 'results_notebook'
os.makedirs(results_dir, exist_ok=True)

model.train()
ema.eval()

total_steps = epochs * len(loader)
train_steps = 0
best_loss = float('inf')
for epoch in range(epochs):
    epoch_loss = 0
    loader_iter = tqdm(loader, desc=f'Epoch {epoch}')
    for step, (x, y) in enumerate(loader_iter):
        x = x.to(device)
        y = y.to(device)
        
        with torch.no_grad():
            x = vae.encode(x).latent_dist.sample().mul_(0.18215)
        
        t = torch.randint(0, diffusion.num_timesteps, (x.shape[0],), device=device)
        current_ilsd_weight = max(0.0, 1.0 - (train_steps / total_steps))
        model_kwargs = dict(y=y, ilsd_weight=current_ilsd_weight, return_ilsd_loops=True)
        
        loss_dict = diffusion.training_losses(model, x, t, model_kwargs)
        loss = loss_dict['loss'].mean()
        
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        
        update_ema(ema, model)
        epoch_loss += loss.item()
        loader_iter.set_postfix({'loss': f'{loss.item():.4f}'})
        train_steps += 1
        
    avg_loss = epoch_loss / len(loader)
    print(f'Epoch {epoch} Average Loss: {avg_loss:.4f}')
    
    if epoch % 10 == 0:
        print(f'Generating samples at epoch {epoch}...')
        with torch.no_grad():
            sampling_diffusion = create_diffusion('250')
            latent_size = config.model.image_size // 8
            n = 20
            z = torch.randn(n, 4, latent_size, latent_size, device=device)
            y = torch.randint(0, config.model.num_classes, (n,), device=device)
            z = torch.cat([z, z], 0)
            y_null = torch.tensor([config.model.num_classes] * n, device=device)
            y = torch.cat([y, y_null], 0)
            model_kwargs = dict(y=y, cfg_scale=4.0, loops=config.sampling.inference_loops)
            samples = sampling_diffusion.p_sample_loop(
                ema.forward_with_cfg, z.shape, z, clip_denoised=False, model_kwargs=model_kwargs, progress=False, device=device
            )
            samples, _ = samples.chunk(2, dim=0)
            samples = vae.decode(samples / 0.18215).sample
            from torchvision.utils import save_image
            save_image(samples, f'{results_dir}/sample_epoch_{epoch}.png', nrow=5, normalize=True, value_range=(-1, 1))
    
    # Save checkpoint if best loss
    if avg_loss < best_loss:
        best_loss = avg_loss
        checkpoint = {
            'model': model.state_dict(),
            'ema': ema.state_dict(),
            'opt': opt.state_dict(),
            'epoch': epoch,
            'best_loss': best_loss
        }
        torch.save(checkpoint, f'{results_dir}/elt-best-model.pt')
        print(f'Saved best model with avg loss {best_loss:.4f}')


Epoch 0: 100%|██████████| 390/390 [00:16<00:00, 23.01it/s, loss=0.1089]


Epoch 0 Average Loss: 0.2723
Generating samples at epoch 0...
Saved best model with avg loss 0.2723


Epoch 1: 100%|██████████| 390/390 [00:17<00:00, 22.74it/s, loss=0.0958]


Epoch 1 Average Loss: 0.1117
Saved best model with avg loss 0.1117


Epoch 2: 100%|██████████| 390/390 [00:17<00:00, 22.70it/s, loss=0.0957]


...

Epoch 597 Average Loss: 0.0518


Epoch 598: 100%|██████████| 390/390 [00:14<00:00, 26.47it/s, loss=0.0464]


Epoch 598 Average Loss: 0.0522


Epoch 599: 100%|██████████| 390/390 [00:15<00:00, 25.55it/s, loss=0.0531]


Epoch 599 Average Loss: 0.0513
Saved best model with avg loss 0.0513


In [7]:
!pip install torch_fidelity

  Using cached torch_fidelity-0.4.0-py3-none-any.whl.metadata (2.1 kB)
Using cached torch_fidelity-0.4.0-py3-none-any.whl (85 kB)


In [5]:
import os
import torch
import torchvision
from tqdm.auto import tqdm
from diffusers.models import AutoencoderKL
import torch_fidelity

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading best checkpoint for evaluation...")
eval_model = ELT(config).to(device)
checkpoint = torch.load("results_notebook/elt-best-model.pt", map_location=device)
eval_model.load_state_dict(checkpoint.get("model", checkpoint))
eval_model.eval()

eval_diffusion = create_diffusion(str(250))
eval_vae = AutoencoderKL.from_pretrained(f"stabilityai/sd-vae-ft-ema").to(device)

# 1. Generate 50k samples
sample_dir = "results_notebook/samples_eval_50k"
os.makedirs(sample_dir, exist_ok=True)

num_samples = 50000
batch_size = 256 # Higher batch size since training is complete!
existing_samples = len(os.listdir(sample_dir))

latent_size = config.model.image_size // 8

if existing_samples >= num_samples:
    print(f"Found {existing_samples} images in '{sample_dir}'. Skipping generation!")
else:
    print(f"Generating {num_samples} samples to {sample_dir} with batch size {batch_size}...")
    num_batches = (num_samples + batch_size - 1) // batch_size
    
    sample_idx = 0
    with torch.no_grad():
        for i in tqdm(range(num_batches), desc="Generating Batches"):
            current_batch_size = min(batch_size, num_samples - sample_idx)
            z = torch.randn(current_batch_size, 4, latent_size, latent_size, device=device)
            y = torch.randint(0, config.model.num_classes, (current_batch_size,), device=device)
            
            z = torch.cat([z, z], 0)
            y_null = torch.tensor([config.model.num_classes] * current_batch_size, device=device)
            y_cfg = torch.cat([y, y_null], 0)
            
            loops = config.sampling.inference_loops
            model_kwargs = dict(y=y_cfg, cfg_scale=4.0, loops=loops)
            
            samples = eval_diffusion.p_sample_loop(
                eval_model.forward_with_cfg, z.shape, z, clip_denoised=False, model_kwargs=model_kwargs, progress=False, device=device
            )
            samples, _ = samples.chunk(2, dim=0)
            samples = eval_vae.decode(samples / 0.18215).sample
            
            for j in range(current_batch_size):
                img = samples[j]
                torchvision.utils.save_image(img, os.path.join(sample_dir, f"{sample_idx:05d}.png"), normalize=True, value_range=(-1, 1))
                sample_idx += 1

# 2. Prepare CIFAR-10 reference dataset
ref_dir = "cifar10_ref"
if not os.path.exists(ref_dir) or len(os.listdir(ref_dir)) < 50000:
    os.makedirs(ref_dir, exist_ok=True)
    print("Downloading CIFAR-10 via Hugging Face for faster speeds...")
    from datasets import load_dataset
    dataset = load_dataset("uoft-cs/cifar10", split="train")
    for i, item in enumerate(tqdm(dataset, desc="Saving Reference Images")):
        item["img"].save(os.path.join(ref_dir, f"{i:05d}.png"))

# 3. Calculate FID and IS
print("Calculating FID and IS...")
metrics = torch_fidelity.calculate_metrics(
    input1=ref_dir,
    input2=sample_dir,
    cuda=torch.cuda.is_available(),
    isc=True,
    fid=True,
    verbose=True
)

print("-" * 50)
print(f"Inception Score (IS): {metrics['inception_score_mean']:.4f} ± {metrics['inception_score_std']:.4f}")
print(f"Frechet Inception Distance (FID): {metrics['frechet_inception_distance']:.4f}")
print("-" * 50)

Loading best checkpoint for evaluation...
Found 50000 images in 'results_notebook/samples_eval_50k'. Skipping generation!
Calculating FID and IS...


Creating feature extractor "inception-v3-compat" with features ['logits_unbiased', '2048']
Extracting features from input1
Looking for samples non-recursivelty in "cifar10_ref" with extensions png,jpg,jpeg
Found 50000 samples
Processing samples:   0%|          | 0/50000 [00:00<?, ?samples/s][W731 12:59:21.840624530 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 360710144 bytes (free: 338100224, total: 16611278848).
[W731 12:59:21.851178396 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 532676608 bytes (free: 197591040, total: 16611278848).
Processing samples:   0%|          | 64/50000 [00:00<03:24, 243.76samples/s][W731 12:59:21.927700539 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 532676608 bytes (free: 19333120, total: 16611278848).
[W731 12:59:21.927737953 CUDACachingAllocator.cpp:3933] memory allocation failed with O

OutOfMemoryError: CUDA out of memory. Tried to allocate 338.00 MiB. GPU 0 has a total capacity of 15.47 GiB of which 18.44 MiB is free. Process 896275 has 13.02 GiB memory in use. Including non-PyTorch memory, this process has 2.38 GiB memory in use. Of the allocated memory 1.45 GiB is allocated by PyTorch, and 672.32 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)